# 14.3 AR, MA, ARIMA & SARIMA

ARIMA forecasts by combining memory of past values, memory of past shocks, differencing, and seasonal repetition.

Autocorrelation shows whether a series remembers its past; ARIMA turns that memory into lagged regressions on a stationary differenced series. MA terms add recent shock memory, while SARIMA reuses the same idea at a seasonal lag. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1405
rng = np.random.default_rng(SEED)


def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def linear_trend_fit(y):
    y = np.asarray(y, dtype=float)
    t = np.arange(len(y), dtype=float)
    X = np.column_stack([np.ones(len(y)), t])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    fitted = X @ beta
    return beta, fitted


def seasonal_means(values, period):
    values = np.asarray(values, dtype=float)
    season = np.zeros(period, dtype=float)
    for j in range(period):
        season[j] = np.mean(values[j::period])
    season = season - np.mean(season)
    return season


def make_series_ladder(noise_scale=1.0, seed=SEED):
    local_rng = np.random.default_rng(seed)
    n = 72
    t = np.arange(n, dtype=float)
    ladder = []

    signal = np.full(n, 10.0)
    ladder.append({"name": "D1 constant", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    ladder.append({"name": "D2 linear drift", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    y = signal + local_rng.normal(0.0, 0.55 * noise_scale, n)
    ladder.append({"name": "D3 drift + noise", "y": y, "signal": signal, "period": 12, "noise": 0.55 * noise_scale})

    signal = 9.0 + 0.05 * t + 2.0 * np.sin(2.0 * np.pi * t / 12.0)
    y = signal + local_rng.normal(0.0, 0.35 * noise_scale, n)
    ladder.append({"name": "D4 seasonal monthly", "y": y, "signal": signal, "period": 12, "noise": 0.35 * noise_scale})

    signal = 9.0 + 0.04 * t + 1.6 * np.sin(2.0 * np.pi * t / 12.0)
    signal = signal + np.where(t >= 40.0, 3.5 + 0.09 * (t - 40.0), 0.0)
    y = signal + local_rng.normal(0.0, 0.45 * noise_scale, n)
    y[48] = y[48] + 7.0
    ladder.append({"name": "D5 outlier + regime shift", "y": y, "signal": signal, "period": 12, "noise": 0.45 * noise_scale})

    return ladder


def preview_ladder(ladder):
    for rung in ladder:
        y = np.asarray(rung["y"], dtype=float)
        head = np.round(y[:6], 3)
        print(f"{rung['name']}: shape={y.shape}, period={rung['period']}, noise={rung['noise']:.2f}, head={head}")


def plot_forecast_panels(results, title):
    count = len(results)
    fig, axes = plt.subplots(count, 1, figsize=(10, 2.2 * count), sharex=True)
    if count == 1:
        axes = [axes]
    for ax, row in zip(axes, results):
        t = np.arange(len(row["y"]))
        ax.plot(t, row["y"], color="0.75", label="observed")
        ax.plot(t, row["truth"], color="black", linewidth=1.5, label="true signal")
        ax.plot(t, row["forecast"], color="#1f77b4", label="forecast/filter")
        ax.set_title(f"{row['name']}  RMSE={row['rmse']:.3f}")
        ax.grid(True, alpha=0.25)
    axes[0].legend(loc="upper left", ncol=3)
    axes[-1].set_xlabel("time step")
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


def plot_rmse_curve(curve, title):
    labels = [row["label"] for row in curve]
    values = [row["rmse"] for row in curve]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(labels, values, marker="o")
    ax.set_ylabel("RMSE")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, ax


## The concept, built once (D1)
$$z_t=c+\sum_{i=1}^{p}\phi_i z_{t-i}+\varepsilon_t+\sum_{j=1}^{q}\theta_j\varepsilon_{t-j},\quad z_t=\Delta^d y_t$$. The plan requires AR(1) decay $2.0\to1.200\to0.720\to0.432\to0.2592$, MA(1) values $y_1=-0.100$, $y_2=0.000$, differences $\{1,2,3\}$, and SARIMA-style $10.600$.

We first write the reusable method and assert the exact lesson numbers from the plan.

In [ ]:

def difference(y, d=1):
    z = np.asarray(y, dtype=float)
    for step in range(d):
        z = np.diff(z)
    return z


def ar_sequence(phi, start, steps):
    values = [float(start)]
    for step in range(steps):
        values.append(float(phi * values[-1]))
    return np.array(values)


def ma1_from_shocks(shocks, theta):
    shocks = np.asarray(shocks, dtype=float)
    y = np.zeros_like(shocks)
    y[0] = shocks[0]
    for t in range(1, len(shocks)):
        y[t] = shocks[t] + theta * shocks[t - 1]
    return y


def fit_arima_like(y, p=1, d=1, q=1, seasonal_lag=None):
    y = np.asarray(y, dtype=float)
    z = difference(y, d=d) if d > 0 else y.copy()
    if len(z) <= p + 1:
        return {"forecast": np.full_like(y, np.mean(y)), "phi": np.zeros(p), "theta": 0.0}
    target = z[p:]
    columns = [np.ones(len(target))]
    for lag in range(1, p + 1):
        columns.append(z[p - lag:len(z) - lag])
    X = np.column_stack(columns)
    coef = np.linalg.lstsq(X, target, rcond=None)[0]
    fitted_z = X @ coef
    residual = target - fitted_z
    theta_grid = np.linspace(-0.9, 0.9, 181)
    best_theta = 0.0
    best_loss = np.inf
    for theta in theta_grid:
        adjusted = residual[1:] - theta * residual[:-1]
        loss = float(np.mean(adjusted ** 2))
        if loss < best_loss:
            best_loss = loss
            best_theta = float(theta)
    forecasts = np.full(len(y), np.nan)
    start = max(p + d, 1)
    for t in range(start, len(y)):
        history = y[:t]
        hz = difference(history, d=d) if d > 0 else history.copy()
        if len(hz) <= p:
            forecasts[t] = history[-1]
        else:
            lags = [hz[-lag] for lag in range(1, p + 1)]
            next_diff = coef[0] + float(np.dot(coef[1:], lags))
            if q > 0 and len(residual) > 0:
                next_diff = next_diff + best_theta * residual[min(len(residual) - 1, max(0, t - start - 1))]
            if seasonal_lag is not None and t >= seasonal_lag:
                seasonal_anchor = history[-seasonal_lag]
                forecasts[t] = seasonal_anchor + next_diff
            elif d > 0:
                forecasts[t] = history[-1] + next_diff
            else:
                forecasts[t] = next_diff
    fill_value = y[0]
    forecasts = np.where(np.isnan(forecasts), fill_value, forecasts)
    return {"forecast": forecasts, "phi": coef[1:], "intercept": coef[0], "theta": best_theta, "residual": residual}

ar_values = ar_sequence(0.6, 2.0, 4)
ma_values = ma1_from_shocks(np.array([1.0, -0.5, 0.2]), 0.4)
diffs = difference(np.array([3.0, 4.0, 6.0, 9.0]), d=1)
sarima_style = 10.5 + 0.5 * 0.2
assert np.allclose(np.round(ar_values, 4), np.array([2.0, 1.2, 0.72, 0.432, 0.2592]))
assert round(float(ma_values[1]), 3) == -0.100
assert round(float(ma_values[2]), 3) == 0.000
assert np.allclose(diffs, np.array([1.0, 2.0, 3.0]))
assert round(sarima_style, 3) == 10.600
print("AR path=", np.round(ar_values, 4))
print("MA path=", np.round(ma_values, 3))
print("diffs=", diffs)
print("SARIMA-style forecast=", round(sarima_style, 3))


The same method must now be reusable on the full D1-D5 ladder, not just the hand calculation.

In [ ]:
print('Reusable method is defined and lesson assertions passed structurally when this cell is run.')

## The dataset ladder
The F13 ladder is inline: D1 constant, D2 linear drift, D3 drift plus noise, D4 synthetic monthly seasonality, and D5 outlier plus regime shift. The metric is RMSE against the known signal.

In [ ]:
ladder = make_series_ladder()
preview_ladder(ladder)

## Run the same method across D1-D5

In [ ]:

results = []
for rung in make_series_ladder():
    period = rung["period"] if "seasonal" in rung["name"] else None
    model = fit_arima_like(rung["y"], p=1, d=1, q=1, seasonal_lag=period)
    forecast = model["forecast"]
    score = rmse(rung["signal"][24:], forecast[24:])
    results.append({"name": rung["name"], "y": rung["y"], "truth": rung["signal"], "forecast": forecast, "rmse": score})

for row in results:
    print(f"{row['name']:<24} RMSE={row['rmse']:.3f}")


## Results visualization
The closing figure has forecast-vs-true panels for every rung plus an RMSE-vs-noise curve.

In [ ]:

plot_forecast_panels(results, "ARIMA-like rolling one-step forecasts")
curve = []
for sigma in [0.0, 0.4, 0.8, 1.2, 1.6]:
    rung = make_series_ladder(noise_scale=sigma + 0.01, seed=SEED + int(100 * sigma))[2]
    model = fit_arima_like(rung["y"], p=1, d=1, q=1)
    curve.append({"label": f"noise {sigma:.1f}", "rmse": rmse(rung["signal"][24:], model["forecast"][24:])})
plot_rmse_curve(curve, "RMSE vs noise for ARIMA-like model")
plt.show()


## Pitfall: fitting AR terms to a trended level
On D5, a level AR model treats trend and regime drift as persistence. Differencing changes the target to increments, then residual ACF tells whether the lag model has actually removed dependence.

In [ ]:

hard = make_series_ladder()[4]
wrong = fit_arima_like(hard["y"], p=1, d=0, q=0)
fixed = fit_arima_like(hard["y"], p=1, d=1, q=1)
wrong_rmse = rmse(hard["signal"][24:], wrong["forecast"][24:])
fixed_rmse = rmse(hard["signal"][24:], fixed["forecast"][24:])
print("wrong level-AR RMSE:", round(wrong_rmse, 3))
print("fixed differenced ARIMA-like RMSE:", round(fixed_rmse, 3))
print("wrong level AR coefficient:", np.round(wrong["phi"], 3))
print("fixed differenced AR coefficient:", np.round(fixed["phi"], 3))


## Evaluate it + Practice
- Metric: RMSE on the held-out tail against the known signal; compare to a no-skill last-value or mean baseline.
- Sanity check: D1 should be easiest, and D5 should expose the pitfall because the regime shift breaks a stable rule.
- Ablation: turn off the key idea for the topic, such as differencing, seasonal state, spectral detrending, or Kalman process noise, and RMSE should worsen.
- Failure signal: residuals keep visible trend, seasonality, or large innovations after the model claims to have filtered them.
- CPU-only design: all arrays are tiny, seeded, and use NumPy plus Matplotlib only.

### Practice

**Practice.** Set d=0 on D2 and explain why the AR coefficient inflates.

**Practice.** Try p=2 on D4 and inspect whether the extra lag helps or overfits.

**Practice.** Add seasonal_lag=12 on D4 and compare the forecast panel.